# DSPy 101 — Replacing Prompts with Signatures

**Week 6 | Notebook 1 of 6**

**What you'll learn:**
- LM configuration (OpenAI + Ollama)
- Your first Signature — QuestionAnswering
- dspy.Predict — basic prediction
- dspy.ChainOfThought — adding rationale
- dspy.ProgramOfThought — math problems with code execution
- Side-by-side output comparison across module types
- Inline vs class-based signature syntax

**Runtime:** ~20 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("06_dspy/01_signatures_modules.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  06_dspy/01_signatures_modules.ipynb
Task:      Signatures and core modules
Calls:     ~15

With GPT-4o:       $0.15 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup & LM Configuration

In [2]:
import dspy

from src.config import get_dspy_lm, print_config

print_config()

# Configure DSPy with our unified LM
lm = get_dspy_lm()
dspy.configure(lm=lm)

print(f"\n✅ DSPy configured with: {lm.model}")

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
LLM_PROVIDER:      openai
  openai model:    gpt-4o
  anthropic model: claude-opus-4-6
  gemini model:    gemini-3.6-flash
  groq model:      openai/gpt-oss-120b
SAMPLE_SIZE:       50
DSPY_TRIALS:       10

✅ DSPy configured with: openai/gpt-4o


## 2. Your First Signature — QuestionAnswering

In [3]:
# Class-based signature (recommended for production)
class QuestionAnswering(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="A concise, factual answer")


# Inline signature (quick prototyping)
# inline_sig = "question -> answer"

print("✅ Signature defined")
print(f"  Input: {QuestionAnswering.fields.keys()}")

✅ Signature defined
  Input: dict_keys(['question', 'answer'])


## 3. dspy.Predict — Basic Prediction

In [4]:
# Basic prediction module
predictor = dspy.Predict(QuestionAnswering)

result = predictor(question="What is the capital of India?")
print("Question: What is the capital of India?")
print(f"Answer: {result.answer}")

Question: What is the capital of India?
Answer: New Delhi


## 4. dspy.ChainOfThought — Adding Rationale

In [5]:
# ChainOfThought automatically adds a 'reasoning' field (dspy 3.x renamed 'rationale')
cot = dspy.ChainOfThought(QuestionAnswering)

result = cot(question="If a train travels 60 km/h for 2.5 hours, how far does it go?")

print("Rationale:")
print(result.reasoning)
print(f"\nAnswer: {result.answer}")

Rationale:
To determine how far the train travels, use the formula for distance: distance = speed × time. Given that the train's speed is 60 km/h and it travels for 2.5 hours, the distance covered is 60 km/h multiplied by 2.5 hours.

Answer: 150 kilometers


## 5. dspy.ProgramOfThought — Math with Code Execution

In [6]:
# ProgramOfThought generates and executes Python code
pot = dspy.ProgramOfThought(QuestionAnswering)

result = pot(question="What is the average of 45, 67, 89, and 23?")

print(f"Answer: {result.answer}")
print("\nThis answer was computed by generated Python code, not guessed by the LLM.")

Answer: 56

This answer was computed by generated Python code, not guessed by the LLM.


## 6. Side-by-Side Output Comparison

In [7]:
question = "Explain transformer attention in one sentence."

modules = {
    "Predict": dspy.Predict(QuestionAnswering),
    "ChainOfThought": dspy.ChainOfThought(QuestionAnswering),
    "ProgramOfThought": dspy.ProgramOfThought(QuestionAnswering),
}

print(f"Question: {question}\n")
for name, module in modules.items():
    result = module(question=question)
    print(f"--- {name} ---")
    if hasattr(result, "reasoning"):
        print(f"Reasoning: {result.reasoning[:100]}...")
    print(f"Answer: {result.answer}\n")

Question: Explain transformer attention in one sentence.

--- Predict ---
Answer: Transformer attention is a mechanism that assigns different importance to different words in a sequence by computing a context-dependent score to enhance the model's ability to capture relationships in the data.

--- ChainOfThought ---
Reasoning: Transformer attention, particularly self-attention, is a mechanism that allows the model to weigh an...
Answer: Transformer attention assigns dynamic weights to different parts of the input sequence to capture dependencies and contextual relevance within the data.



--- ProgramOfThought ---
Reasoning: The `final_generated_code` provides a concise definition of transformer attention, highlighting its ...
Answer: Transformer attention calculates a weighted representation of inputs by considering relationships between all parts of the sequence through learned attention scores.



## 7. Inline vs Class-Based Signature Syntax

In [8]:
# Class-based (recommended)
class SentimentClassification(dspy.Signature):
    """Classify the sentiment of the given text."""

    text: str = dspy.InputField()
    sentiment: str = dspy.OutputField(desc="positive, negative, or neutral")
    confidence: float = dspy.OutputField(desc="confidence score between 0 and 1")


classifier = dspy.Predict(SentimentClassification)
result = classifier(text="This product is amazing!")
print(f"Class-based: {result.sentiment} (confidence: {result.confidence})")

# Inline (quick prototyping)
inline_classifier = dspy.Predict("text -> sentiment, confidence")
result2 = inline_classifier(text="This product is amazing!")
print(f"Inline: {result2.sentiment} (confidence: {result2.confidence})")

Class-based: positive (confidence: 0.95)


Inline: positive (confidence: high)


## 8. Exercise: Build a Sentiment + Topic Classifier Signature

Create a signature that classifies both sentiment AND topic in one call.

In [9]:
# YOUR TURN: Define a multi-output signature

# class SentimentTopicClassifier(dspy.Signature):
#     """Classify sentiment and detect topic."""
#     text: str = dspy.InputField()
#     sentiment: str = dspy.OutputField()
#     topic: str = dspy.OutputField()
#     confidence: float = dspy.OutputField()

# cls = dspy.Predict(SentimentTopicClassifier)
# result = cls(text="Your text here")
# print(result)

---

**Next:** [02_optimizers.ipynb](02_optimizers.ipynb) — BootstrapFewShot vs MIPROv2